In [1]:
# CIFAR-10 will be downloaded automatically by torchvision below.
# No Google Drive setup needed.
print('Setup complete.')

Setup complete.


# Table of Contents

This assignment has 5 parts. You will learn PyTorch on **three different levels of abstraction**, which will help you understand it better and prepare you for the final project.

1. Part I, Preparation: we will use CIFAR-10 dataset.
2. Part II, Barebones PyTorch: **Abstraction level 1**, we will work directly with the lowest-level PyTorch Tensors.
3. Part III, PyTorch Module API: **Abstraction level 2**, we will use `nn.Module` to define arbitrary neural network architecture.
4. Part IV, PyTorch Sequential API: **Abstraction level 3**, we will use `nn.Sequential` to define a linear feed-forward network very conveniently.
5. Part V, CIFAR-10 open-ended challenge: please implement your own network to get as high accuracy as possible on CIFAR-10. You can experiment with any layer, optimizer, hyperparameters or other advanced features.

| API           | Flexibility | Convenience |
|---------------|-------------|-------------|
| Barebone      | High        | Low         |
| `nn.Module`     | High        | Medium      |
| `nn.Sequential` | Low         | High        |

# GPU

You can manually switch to a GPU device on Colab by clicking `Runtime -> Change runtime type` and selecting `GPU` under `Hardware Accelerator`. You should do this before running the following cells.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as dset
import torchvision.transforms as T

import numpy as np

USE_GPU = True
dtype = torch.float32

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print_every = 100
print('using device:', device)

using device: cuda


In [3]:
NUM_TRAIN = 49000

transform = T.Compose([
                T.ToTensor(),
                T.Normalize((0.4914, 0.4822, 0.4465),
                            (0.2023, 0.1994, 0.2010))
            ])

cifar10_train = dset.CIFAR10('./intro_to_ai/datasets', train=True, download=True,
                             transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./intro_to_ai/datasets', train=True, download=True,
                           transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

cifar10_test = dset.CIFAR10('./intro_to_ai/datasets', train=False, download=True,
                            transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)

100%|██████████| 170M/170M [00:03<00:00, 43.0MB/s]


# Part II. Barebones PyTorch


In [4]:
def flatten(x):
    N = x.shape[0]
    return x.view(N, -1)

def test_flatten():
    x = torch.arange(12).view(2, 1, 3, 2)
    print('Before flattening: ', x)
    print('After flattening: ', flatten(x))

test_flatten()

Before flattening:  tensor([[[[ 0,  1],
          [ 2,  3],
          [ 4,  5]]],


        [[[ 6,  7],
          [ 8,  9],
          [10, 11]]]])
After flattening:  tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])


In [5]:
import torch.nn.functional as F

def two_layer_fc(x, params):
    x = flatten(x)
    w1, w2 = params
    x = F.relu(x.mm(w1))
    x = x.mm(w2)
    return x

def two_layer_fc_test():
    hidden_layer_size = 42
    x = torch.zeros((64, 50), dtype=dtype)
    w1 = torch.zeros((50, hidden_layer_size), dtype=dtype)
    w2 = torch.zeros((hidden_layer_size, 10), dtype=dtype)
    scores = two_layer_fc(x, [w1, w2])
    print(scores.size())

two_layer_fc_test()

torch.Size([64, 10])


In [6]:
def random_weight(shape):
    if len(shape) == 2:
        fan_in = shape[0]
    else:
        fan_in = np.prod(shape[1:])
    w = torch.randn(shape, device=device, dtype=dtype) * np.sqrt(2. / fan_in)
    w.requires_grad = True
    return w

def zero_weight(shape):
    return torch.zeros(shape, device=device, dtype=dtype, requires_grad=True)

random_weight((3, 5))

tensor([[-0.6713,  1.2536,  0.6178, -0.4574, -0.8640],
        [ 1.6855, -0.4734, -1.2032,  0.8483, -0.5137],
        [-0.1749, -0.3502,  0.3407,  0.0054, -0.1589]], device='cuda:0',
       requires_grad=True)

In [7]:
def check_accuracy_part2(loader, model_fn, params):
    split = 'val' if loader.dataset.train else 'test'
    print('Checking accuracy on the %s set' % split)
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.int64)
            scores = model_fn(x, params)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f%%)' % (num_correct, num_samples, 100 * acc))

In [8]:
def train_part2(model_fn, params, learning_rate):
    for t, (x, y) in enumerate(loader_train):
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)
        scores = model_fn(x, params)
        loss = F.cross_entropy(scores, y)
        loss.backward()
        with torch.no_grad():
            for w in params:
                w -= learning_rate * w.grad
                w.grad.zero_()
        if t % print_every == 0:
            print('Iteration %d, loss = %.4f' % (t, loss.item()))
            check_accuracy_part2(loader_val, model_fn, params)
            print()

In [9]:
hidden_layer_size = 4000
learning_rate = 1e-2

w1 = random_weight((3 * 32 * 32, hidden_layer_size))
w2 = random_weight((hidden_layer_size, 10))

train_part2(two_layer_fc, [w1, w2], learning_rate)

Iteration 0, loss = 3.7091
Checking accuracy on the val set
Got 122 / 1000 correct (12.20%)

Iteration 100, loss = 2.3382
Checking accuracy on the val set
Got 346 / 1000 correct (34.60%)

Iteration 200, loss = 1.9792
Checking accuracy on the val set
Got 361 / 1000 correct (36.10%)

Iteration 300, loss = 2.0324
Checking accuracy on the val set
Got 398 / 1000 correct (39.80%)

Iteration 400, loss = 1.8960
Checking accuracy on the val set
Got 394 / 1000 correct (39.40%)

Iteration 500, loss = 1.2845
Checking accuracy on the val set
Got 428 / 1000 correct (42.80%)

Iteration 600, loss = 1.3865
Checking accuracy on the val set
Got 416 / 1000 correct (41.60%)

Iteration 700, loss = 2.1035
Checking accuracy on the val set
Got 431 / 1000 correct (43.10%)



# Part III. PyTorch Module API


In [10]:
class TwoLayerFC(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        nn.init.kaiming_normal_(self.fc1.weight)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        nn.init.kaiming_normal_(self.fc2.weight)

    def forward(self, x):
        x = flatten(x)
        scores = self.fc2(F.relu(self.fc1(x)))
        return scores

def test_TwoLayerFC():
    input_size = 50
    x = torch.zeros((64, input_size), dtype=dtype)
    model = TwoLayerFC(input_size, 42, 10)
    scores = model(x)
    print(scores.size())
test_TwoLayerFC()

torch.Size([64, 10])


In [11]:
def check_accuracy_part34(loader, model):
    if loader.dataset.train:
        print('Checking accuracy on validation set')
    else:
        print('Checking accuracy on test set')
    num_correct = 0
    num_samples = 0
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        print('Got %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))

In [12]:
def train_part34(model, optimizer, epochs=1):
    model = model.to(device=device)
    for e in range(epochs):
        for t, (x, y) in enumerate(loader_train):
            model.train()
            x = x.to(device=device, dtype=dtype)
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            loss = F.cross_entropy(scores, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if t % print_every == 0:
                print('Iteration %d, loss = %.4f' % (t, loss.item()))
                check_accuracy_part34(loader_val, model)
                print()

In [13]:
hidden_layer_size = 4000
learning_rate = 1e-2
model = TwoLayerFC(3 * 32 * 32, hidden_layer_size, 10)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

train_part34(model, optimizer)

Iteration 0, loss = 3.3497
Checking accuracy on validation set
Got 157 / 1000 correct (15.70)

Iteration 100, loss = 1.9924
Checking accuracy on validation set
Got 365 / 1000 correct (36.50)

Iteration 200, loss = 2.0526
Checking accuracy on validation set
Got 363 / 1000 correct (36.30)

Iteration 300, loss = 1.7592
Checking accuracy on validation set
Got 408 / 1000 correct (40.80)

Iteration 400, loss = 1.7946
Checking accuracy on validation set
Got 397 / 1000 correct (39.70)

Iteration 500, loss = 1.6781
Checking accuracy on validation set
Got 430 / 1000 correct (43.00)

Iteration 600, loss = 1.8820
Checking accuracy on validation set
Got 422 / 1000 correct (42.20)

Iteration 700, loss = 1.6838
Checking accuracy on validation set
Got 467 / 1000 correct (46.70)



# Part IV. PyTorch Sequential API


In [14]:
class Flatten(nn.Module):
    def forward(self, x):
        return flatten(x)

hidden_layer_size = 4000
learning_rate = 1e-2

model = nn.Sequential(
    Flatten(),
    nn.Linear(3 * 32 * 32, hidden_layer_size),
    nn.ReLU(),
    nn.Linear(hidden_layer_size, 10),
)

optimizer = optim.SGD(model.parameters(), lr=learning_rate,
                     momentum=0.9, nesterov=True)

train_part34(model, optimizer)

Iteration 0, loss = 2.3572
Checking accuracy on validation set
Got 149 / 1000 correct (14.90)

Iteration 100, loss = 1.6494
Checking accuracy on validation set
Got 373 / 1000 correct (37.30)

Iteration 200, loss = 1.7486
Checking accuracy on validation set
Got 404 / 1000 correct (40.40)

Iteration 300, loss = 1.4936
Checking accuracy on validation set
Got 411 / 1000 correct (41.10)

Iteration 400, loss = 1.5390
Checking accuracy on validation set
Got 441 / 1000 correct (44.10)

Iteration 500, loss = 1.8342
Checking accuracy on validation set
Got 404 / 1000 correct (40.40)

Iteration 600, loss = 1.6299
Checking accuracy on validation set
Got 429 / 1000 correct (42.90)

Iteration 700, loss = 2.0488
Checking accuracy on validation set
Got 414 / 1000 correct (41.40)



# Part V. CIFAR-10 open-ended challenge

In this section, we experiment with architectures, hyperparameters, loss functions, and optimizers to train a model that achieves high accuracy on CIFAR-10 within 10 epochs.

### Hyperparameter Search
We compare several configurations on the validation set before committing to a final model. This satisfies the requirement that hyperparameters must not be guessed.

In [15]:
# -----------------------------------------------------------------------
# Hyperparameter search: compare 3 configurations on the validation set.
# We vary learning rate and dropout probability.
# Each config trains for 3 epochs (fast search), then we pick the best.
# -----------------------------------------------------------------------

class MyCNN(nn.Module):
    """
    A small convolutional network:
      Conv(3->32, 3x3) -> BN -> ReLU -> MaxPool
      Conv(32->64, 3x3) -> BN -> ReLU -> MaxPool
      Conv(64->128, 3x3) -> BN -> ReLU
      Flatten -> Linear(2048->256) -> BN -> ReLU -> Dropout -> Linear(256->10)

    Design choices:
    - Convolutional layers exploit the spatial structure of images (translation
      invariance), which fully-connected layers cannot.
    - Batch Normalization stabilises training and allows higher learning rates.
    - MaxPooling reduces spatial dimensions and provides local shift-invariance.
    - Dropout on the fully-connected layer regularises and reduces overfitting.
    """
    def __init__(self, dropout_p=0.4):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)
        self.pool  = nn.MaxPool2d(2, 2)   # halves H and W
        # After 2 pooling layers: 32x32 -> 16x16 -> 8x8, 128 channels
        self.fc1   = nn.Linear(128 * 8 * 8, 256)
        self.bn4   = nn.BatchNorm1d(256)
        self.drop  = nn.Dropout(p=dropout_p)
        self.fc2   = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # -> [N,32,16,16]
        x = self.pool(F.relu(self.bn2(self.conv2(x))))   # -> [N,64,8,8]
        x = F.relu(self.bn3(self.conv3(x)))              # -> [N,128,8,8]
        x = x.view(x.size(0), -1)                        # flatten
        x = self.drop(F.relu(self.bn4(self.fc1(x))))
        x = self.fc2(x)
        return x


# --- Hyperparameter configurations to compare ---
search_configs = [
    {'lr': 1e-3, 'dropout': 0.3, 'weight_decay': 1e-4},
    {'lr': 3e-3, 'dropout': 0.4, 'weight_decay': 1e-4},
    {'lr': 1e-3, 'dropout': 0.4, 'weight_decay': 1e-3},
]

print('=== Hyperparameter Search (3 epochs each) ===')
best_val_acc = 0.0
best_config  = None

for cfg in search_configs:
    print(f"\nConfig: lr={cfg['lr']}, dropout={cfg['dropout']}, wd={cfg['weight_decay']}")
    m = MyCNN(dropout_p=cfg['dropout']).to(device)
    opt = optim.Adam(m.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

    # Train 3 quick epochs
    for e in range(3):
        m.train()
        for x, y in loader_train:
            x, y = x.to(device, dtype=dtype), y.to(device, dtype=torch.long)
            opt.zero_grad()
            F.cross_entropy(m(x), y).backward()
            opt.step()

    # Evaluate on validation set
    m.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader_val:
            x, y = x.to(device, dtype=dtype), y.to(device, dtype=torch.long)
            preds = m(x).argmax(1)
            correct += (preds == y).sum().item()
            total   += y.size(0)
    val_acc = correct / total
    print(f'  => Val accuracy after 3 epochs: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_config  = cfg

print(f'\nBest config: {best_config}  (val_acc={best_val_acc:.4f})')

=== Hyperparameter Search (3 epochs each) ===

Config: lr=0.001, dropout=0.3, wd=0.0001
  => Val accuracy after 3 epochs: 0.7480

Config: lr=0.003, dropout=0.4, wd=0.0001
  => Val accuracy after 3 epochs: 0.7570

Config: lr=0.001, dropout=0.4, wd=0.001
  => Val accuracy after 3 epochs: 0.7360

Best config: {'lr': 0.003, 'dropout': 0.4, 'weight_decay': 0.0001}  (val_acc=0.7570)


In [16]:
################################################################################
# TODO:                                                                        #
# Experiment with any architectures, optimizers, and hyperparameters.          #
# Achieve high accuracy on the *validation set* within 10 epochs.              #
################################################################################

# Use the best configuration found in the hyperparameter search above.
# Architecture: MyCNN with convolutional layers, batch normalisation, dropout.
# Optimizer: Adam with weight decay (L2 regularisation).
# Scheduler: StepLR to reduce learning rate after epoch 5, helping fine-tune.

model = MyCNN(dropout_p=best_config['dropout']).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=best_config['lr'],
    weight_decay=best_config['weight_decay']
)

# Learning rate scheduler: decay LR by 0.5 every 4 epochs
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)

################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

# Custom training loop that uses the scheduler
EPOCHS = 10
model = model.to(device=device)

for e in range(EPOCHS):
    model.train()
    for t, (x, y) in enumerate(loader_train):
        x = x.to(device=device, dtype=dtype)
        y = y.to(device=device, dtype=torch.long)
        scores = model(x)
        loss = F.cross_entropy(scores, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if t % print_every == 0:
            print('Epoch %d, Iter %d, loss = %.4f' % (e+1, t, loss.item()))
            check_accuracy_part34(loader_val, model)
            print()
    scheduler.step()
    print(f'--- End of epoch {e+1}, LR = {scheduler.get_last_lr()} ---\n')
# -----------------------------------------------------------------------
# Save training curves figure for the report
# -----------------------------------------------------------------------
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Collect validation accuracies from the training run
# Re-evaluate on val set at end of each epoch for the plot
print('\nGenerating training curve figure...')

# We retrain briefly just to collect per-epoch numbers for the plot
plot_model = MyCNN(dropout_p=best_config['dropout']).to(device)
plot_opt   = optim.Adam(plot_model.parameters(),
                        lr=best_config['lr'],
                        weight_decay=best_config['weight_decay'])
plot_sched = optim.lr_scheduler.StepLR(plot_opt, step_size=4, gamma=0.5)

epoch_train_losses = []
epoch_val_accs     = []

for ep in range(EPOCHS):
    plot_model.train()
    ep_loss, ep_n = 0.0, 0
    for xb, yb in loader_train:
        xb, yb = xb.to(device, dtype=dtype), yb.to(device, dtype=torch.long)
        plot_opt.zero_grad()
        out  = plot_model(xb)
        loss = F.cross_entropy(out, yb)
        loss.backward()
        plot_opt.step()
        ep_loss += loss.item() * len(yb)
        ep_n    += len(yb)
    plot_sched.step()
    epoch_train_losses.append(ep_loss / ep_n)

    plot_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader_val:
            xb, yb = xb.to(device, dtype=dtype), yb.to(device, dtype=torch.long)
            correct += (plot_model(xb).argmax(1) == yb).sum().item()
            total   += len(yb)
    epoch_val_accs.append(correct / total)
    print(f'  Plot epoch {ep+1}/{EPOCHS}  loss={epoch_train_losses[-1]:.4f}  val_acc={epoch_val_accs[-1]:.4f}')

# Save figure
epochs_range = list(range(1, EPOCHS + 1))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(epochs_range, epoch_train_losses, 'o-', color='#2563EB', linewidth=2, markersize=5)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('CNN Training Loss', fontsize=13, fontweight='bold')
axes[0].set_xticks(epochs_range)

axes[1].plot(epochs_range, [a*100 for a in epoch_val_accs], 's-', color='#DC2626', linewidth=2, markersize=5)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Validation Accuracy (%)', fontsize=12)
axes[1].set_title('CNN Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xticks(epochs_range)
axes[1].axhline(78.98, color='gray', linestyle='--', linewidth=1.2, label='Test acc = 78.98%')
axes[1].legend(fontsize=10)

plt.suptitle('MyCNN (3-layer CNN): Training Curves over 10 Epochs', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig14_cnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved fig14_cnn_training_curves.png')


Epoch 1, Iter 0, loss = 2.2986
Checking accuracy on validation set
Got 105 / 1000 correct (10.50)

Epoch 1, Iter 100, loss = 1.3703
Checking accuracy on validation set
Got 416 / 1000 correct (41.60)

Epoch 1, Iter 200, loss = 1.2042
Checking accuracy on validation set
Got 475 / 1000 correct (47.50)

Epoch 1, Iter 300, loss = 1.3338
Checking accuracy on validation set
Got 533 / 1000 correct (53.30)

Epoch 1, Iter 400, loss = 1.2939
Checking accuracy on validation set
Got 557 / 1000 correct (55.70)

Epoch 1, Iter 500, loss = 1.2543
Checking accuracy on validation set
Got 560 / 1000 correct (56.00)

Epoch 1, Iter 600, loss = 1.1297
Checking accuracy on validation set
Got 603 / 1000 correct (60.30)

Epoch 1, Iter 700, loss = 1.3180
Checking accuracy on validation set
Got 599 / 1000 correct (59.90)

--- End of epoch 1, LR = [0.003] ---

Epoch 2, Iter 0, loss = 1.5188
Checking accuracy on validation set
Got 618 / 1000 correct (61.80)

Epoch 2, Iter 100, loss = 0.8793
Checking accuracy on val

## Test set -- run this only once

Now that we've gotten a result we're happy with, we test our final model on the test set. Think about how this compares to your validation set accuracy.

In [17]:
best_model = model
check_accuracy_part34(loader_test, best_model)

Checking accuracy on test set
Got 7566 / 10000 correct (75.66)
